# Furniture Manual Segmentation API

This notebook outlines the backend implementation for processing a PDF
assembly manual into a sequence of individual step images using an AI model
and computer‑vision logic.  The goal is to translate Nano Banana Pro’s
visual annotations into precise pixel coordinates for cropping the clean
source images.

In [1]:
# install dependencies so that cv2 etc. are available
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python", "pdf2image", "requests"], check=True)
print("packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 47.1 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 57.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 62.7 MB/s  0:00:00
   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/4 [numpy]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [opencv-python]0m [opencv-python]
packages installed



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [11]:

# install system requirement needed by OpenCV (libGL)
!apt-get update && apt-get install -y libgl1-mesa-glx poppler-utils


Reading package lists... Done
E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)


In [ ]:

# install python-multipart required by FastAPI testclient form handling
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "python-multipart"], check=True)


In [9]:
# quick smoke test of processing logic (without Replicate token)
import sys, importlib
sys.path.append('/workspaces/wayfair_studio_backend')
import services.manual_processor
importlib.reload(services.manual_processor)
from services.manual_processor import start_manual_processing, get_job_status
from pathlib import Path
from PIL import Image
import time

# create a trivial PDF
img = Image.new('RGB',(100,100),'white')
pdf_path = '/tmp/test_manual.pdf'
img.save(pdf_path,'PDF')
job = start_manual_processing(Path(pdf_path), name='Test import')
print('started', job)
while True:
    status = get_job_status(job)
    print('status', status)
    if status['status'] != 'processing':
        break
    time.sleep(0.5)
print('final', status)


started 11513652-6b72-499e-b79a-13edb3ba7216
status {'status': 'processing', 'manual_id': 1001, 'step_count': 0, 'error': None}
status {'status': 'failed', 'manual_id': 1001, 'step_count': 0, 'error': 'Unable to get page count. Is poppler installed and in PATH?'}
final {'status': 'failed', 'manual_id': 1001, 'step_count': 0, 'error': 'Unable to get page count. Is poppler installed and in PATH?'}


In [5]:
# replace opencv with headless build (no libGL requirement)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "opencv-python-headless"], check=True)
import cv2
print("cv2 version", cv2.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 53.9 MB/s  0:00:016m0:00:0100:01
cv2 version 4.13.0



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [13]:

# simple TestClient demonstration
from fastapi.testclient import TestClient
from main import app

client = TestClient(app)

with open('/tmp/test_manual.pdf','rb') as f:
    resp = client.post('/api/v1/manuals/process', files={'file': ('test.pdf', f, 'application/pdf')})
    print('endpoint response', resp.status_code, resp.json())


endpoint response 400 {'detail': 'There was an error parsing the body'}


In [12]:
# ensure python-multipart is available for form uploads
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "python-multipart"], check=True)
print("multipart installed")

multipart installed



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip



```python
# attempt to install required packages
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python", "pdf2image", "requests"], check=True)
print("install complete")
```


## 1. API Endpoint Implementation

Define a FastAPI (or Flask) endpoint that receives a PDF via multipart/form-data.  The handler
stores the upload somewhere temporary and kicks off processing, returning a job id for polling.

```python
from fastapi import FastAPI, UploadFile, File, Form
import tempfile
from pathlib import Path

app = FastAPI()

@app.post("/api/v1/manuals/process")
def process_manual_endpoint(file: UploadFile = File(...), name: str = Form(None)):
    tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False)
    tmp.write(file.file.read())
    tmp.flush()
    job_id = start_manual_processing(Path(tmp.name), name)
    return {"job_id": job_id, "status": "processing"}
```

## 2. PDF Ingestion and Storage

Uploaded files are written to a temporary directory (could be an S3 "processing"
bucket in production).  We keep the path around so the background worker can read
it later.

```python
# shown above in the endpoint example; we simply persist the bytes to disk
```

## 3. PDF to Image Conversion (300 DPI)

We use `pdf2image.convert_from_path` with `dpi=300` to get a list of PIL
`Image` objects.  These images are saved to temporary PNG files for later
vision steps.

```python
from pdf2image import convert_from_path

pages = convert_from_path(str(pdf_path), dpi=300)
for i,page in enumerate(pages):
    page.save(f"/tmp/page_{i}.png", "PNG")
```

## 4. Manuals Table Initialization

When a job begins we insert a record into the `manuals` table with
status `PROCESSING`.  This ensures the frontend can query metadata and the
entry exists for the later step rows.

```python
conn = _get_connection()
cur = conn.cursor()
cur.execute("ALTER TABLE manuals ADD COLUMN IF NOT EXISTS status TEXT")
cur.execute(
    "INSERT INTO manuals (name, slug, description, status) VALUES (%s,%s,%s,%s) RETURNING id",
    (name, slug, description, "PROCESSING"),
)
manual_id = cur.fetchone()[0]
conn.commit()
```


## 5. AI Model Integration (Nano Banana Pro)

Each clean page is fed to the Nano Banana Pro model via Replicate.  The model
returns a new image with coloured bounding boxes drawn around each step.  We
store the annotated version temporarily for later analysis.

```python
if os.getenv("REPLICATE_API_TOKEN"):
    with open(page_path, "rb") as f:
        output = replicate.run(NANO_MODEL, input={"image": f})
    # output often a URL pointing to the generated image
    annotated_url = output[0] if isinstance(output, list) else str(output)
    download_file(annotated_url, annotated_path)
else:
    annotated_path = page_path  # no-op
```

## 6. Annotated Image Handling

Both the original PNG and the annotated variant are held on disk (or in
memory) until the cropping logic completes.  We do **not** apply the AI’s
boxes to the clean image when cropping, so the final steps remain un‑marked.

```python
# paths already created above: page_path and annotated_path
# later we will pass them to the CV extraction function
```

## 7. Bounding Box Extraction via Color Masking

A simple but robust approach is to subtract the annotated image from the
original so that only the drawn lines remain.  Convert the difference to
HSV and apply a mask to any non‑zero hue – this sidesteps the need to know
the box colour ahead of time.

```python
orig = cv2.imread(str(page_path))
annot = cv2.imread(str(annotated_path))
diff = cv2.absdiff(orig, annot)
gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)
```

## 8. Contour Detection and Geometry Mapping

Find contours on the thresholded mask and compute bounding rectangles around
them.  Each rect corresponds to a candidate step.

```python
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
boxes = [cv2.boundingRect(c) for c in contours]
```

## 9. Redundancy Filtering of Bounding Boxes

Discard any box that is more than 90% contained within another; this filters
out tiny annotations like screws.

```python
filtered = []
for b in boxes:
    x,y,w,h = b
    if not any(
        x>=ox and y>=oy and x+w <= ox+ow and y+h <= oy+oh and (w*h) <= (ow*oh)*0.9
        for ox,oy,ow,oh in boxes if (ox,oy,ow,oh) != b
    ):
        filtered.append(b)
```

## 10. Cropping Step Images from Original PNG

Use the filtered coordinates to crop the clean page image – this ensures
there are no coloured boxes on the final step assets.

```python
for idx,(x,y,w,h) in enumerate(filtered, start=1):
    crop = orig[y:y+h, x:x+w]
    out_path = manual_dir / f"step{idx}.png"
    cv2.imwrite(str(out_path), crop)
```

## 11. Logical Sorting of Steps

After gathering crops from all pages we sort them first by page index then by
their y‑coordinate and x‑coordinate.

```python
all_boxes.sort(key=lambda item: (item.page, item.y, item.x))
```

## 12. Uploading Cropped Steps to Cloud Storage

In a production environment the cropped PNGs would be pushed to S3 (or
equivalent) and the resulting URL stored in the database.  For simplicity we
write them under `public/manuals/<id>` where they are served statically.

```python
# pseudo-code / placeholder
for step_path in manual_dir.iterdir():
    s3_url = upload_to_s3(step_path)
    record_image_url(manual_id, step_num, s3_url)
```

## 13. Database Insertion for Steps

After each crop is saved we insert a corresponding row into the `steps`
table and link it to the manual.  `ensure_manual_and_step` is a helper which
creates the manual row if necessary and does an upsert on the step itself.

```python
image_url = f"{base_url}/manuals/{manual_id}/step{n}.png"
ensure_manual_and_step(manual_id, n, image_url)
```

## 14. Manuals Table Status Update

Once all pages have been processed and step records inserted we mark the
manual as `COMPLETED` in the database.  This makes it easy for the UI to
filter or display progress.

```python
cur.execute("UPDATE manuals SET status = %s WHERE id = %s", ("COMPLETED", manual_id))
```

## 15. Webhook/Response Handling

The endpoint that kicked off the job returns a job ID; once the background
worker completes it stores `step_count` and `manual_id` in the job record.
Clients can poll the status endpoint or we could additionally POST to a
configurable webhook URL.

```python
@app.get("/api/v1/manuals/process/{job_id}")
def get_process_status(job_id: str):
    return JOBS[job_id]
```


## 16. Error Handling and Logging

- **No boxes detected** – we log a warning and treat the entire page as one step.
- **AI timeout** – the endpoint always returns a job ID so the frontend can poll;
  the background thread can handle long-running calls without blocking the
  request.
- **Missing API token** – the pipeline gracefully degrades by skipping the AI
  pass and using each page as a single step.
- **Unexpected exceptions** – captured in the job record so the UI can display
  an error message.

The worker updates the in‑memory `JOBS` dictionary with `status: "failed"`
and an `error` string when something goes wrong.
